In [0]:
# 1. FORCE INSTALL REQUIRED LIBRARY
%pip install lightgbm

import pandas as pd
import numpy as np
import lightgbm as lgb
from datetime import timedelta
import itertools
import os

# ==========================================
# 2. SMART DATA LOADER
# ==========================================
possible_paths = [
    "/Volumes/workspace/krishigati/data_landing/Agriculture_price_dataset.csv",
    "Agriculture_price_dataset.csv", 
    "Agriculture_price_dataset (1).csv",
    "/dbfs/FileStore/tables/Agriculture_price_dataset.csv",
    "/dbfs/FileStore/tables/Agriculture_price_dataset__1_.csv"
]

df = None
for path in possible_paths:
    if os.path.exists(path):
        print(f"Found dataset at: {path}")
        df = pd.read_csv(path)
        break

if df is None:
    raise FileNotFoundError("Dataset not found! Please check your file paths.")

# ==========================================
# 3. CLEANING & FILTERING FOR NASHIK
# ==========================================
df.rename(columns={
    'District Name': 'District',
    'Market Name': 'Mandi',
    'Price Date': 'Date',
    'Modal_Price': 'Price'
}, inplace=True)

df['District'] = df['District'].astype(str).str.lower().str.strip()
df['Commodity'] = df['Commodity'].astype(str).str.title().str.strip()

nashik_df = df[
    (df['District'] == 'nashik') & 
    (df['Commodity'].isin(['Onion', 'Potato', 'Wheat']))
].copy()

if nashik_df.empty:
    raise ValueError("The filtered dataset is empty! Could not find Nashik data.")

# ==========================================
# 4. TIME-SERIES FEATURE ENGINEERING
# ==========================================
nashik_df['Date'] = pd.to_datetime(nashik_df['Date'], format='mixed', errors='coerce')
nashik_df = nashik_df.dropna(subset=['Date', 'Price'])

nashik_df['Year'] = nashik_df['Date'].dt.year
nashik_df['Month'] = nashik_df['Date'].dt.month
nashik_df['Day'] = nashik_df['Date'].dt.day
nashik_df['DayOfWeek'] = nashik_df['Date'].dt.dayofweek

mandi_categories = list(nashik_df['Mandi'].unique())
commodity_categories = ['Onion', 'Potato', 'Wheat']

nashik_df['Mandi'] = pd.Categorical(nashik_df['Mandi'], categories=mandi_categories)
nashik_df['Commodity'] = pd.Categorical(nashik_df['Commodity'], categories=commodity_categories)

features = ['Mandi', 'Commodity', 'Year', 'Month', 'Day', 'DayOfWeek']
target = 'Price'

X_train = nashik_df[features]
y_train = nashik_df[target]

# ==========================================
# 5. TRAIN THE FORECASTING MODEL
# ==========================================
print(f"Training LightGBM AI Model on {len(X_train)} records...")
model = lgb.LGBMRegressor(
    n_estimators=150, 
    learning_rate=0.05, 
    random_state=42, 
    n_jobs=-1 
)
model.fit(X_train, y_train)

# ==========================================
# 6. PREDICT THE NEXT 4 DAYS
# ==========================================
today = pd.Timestamp.today().normalize()
future_dates = [today + timedelta(days=i) for i in range(1, 5)]

future_combinations = list(itertools.product(mandi_categories, commodity_categories, future_dates))
future_df = pd.DataFrame(future_combinations, columns=['Mandi', 'Commodity', 'Date'])

future_df['Year'] = future_df['Date'].dt.year
future_df['Month'] = future_df['Date'].dt.month
future_df['Day'] = future_df['Date'].dt.day
future_df['DayOfWeek'] = future_df['Date'].dt.dayofweek

future_df['Mandi'] = pd.Categorical(future_df['Mandi'], categories=mandi_categories)
future_df['Commodity'] = pd.Categorical(future_df['Commodity'], categories=commodity_categories)

X_predict = future_df[features]
future_df['Predicted_Price'] = model.predict(X_predict).round(2)

# ==========================================
# 7. FORMAT & SAVE TO DATABRICKS CATALOG
# ==========================================
output_df = future_df[['Date', 'Mandi', 'Commodity', 'Predicted_Price']]
output_df = output_df.sort_values(by=['Date', 'Commodity', 'Mandi'])

# Convert the Pandas DataFrame to a Spark DataFrame
spark_df = spark.createDataFrame(output_df)

# Define your Unity Catalog table path
# (Note: I used 'krishigati' based on your file path, change it to 'kishigati' if your schema is spelled differently)
table_name = "workspace.krishigati.predicted_mandi_prices"

# Write to the catalog as a Delta Table (Overwrites previous predictions so it stays fresh)
spark_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

print(f"\nSUCCESS! Predictions generated for {len(mandi_categories)} Mandis.")
print(f"Data successfully saved to Delta Table: {table_name}")

# Display as an interactive Databricks table
display(spark_df)

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Found dataset at: /Volumes/workspace/krishigati/data_landing/Agriculture_price_dataset.csv
Training LightGBM AI Model on 9555 records...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000120 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 84
[LightGBM] [Info] Number of data points in the train set: 9555, number of used features: 6
[LightGBM] [Info] Start training from score 2283.634491

SUCCESS! Predictions generated for 25 Mandis.
Data successfully saved to Delta Table: workspace.krishigati.predicted_mandi_prices


Date,Mandi,Commodity,Predicted_Price
2026-03-29T00:00:00.000Z,Lasalgaon(Niphad),Onion,1469.78
2026-03-29T00:00:00.000Z,Lasalgaon(Vinchur),Onion,1455.03
2026-03-29T00:00:00.000Z,Pimpalgaon,Onion,1452.47
2026-03-29T00:00:00.000Z,Malegaon,Onion,1459.39
2026-03-29T00:00:00.000Z,Nandgaon,Onion,1316.55
2026-03-29T00:00:00.000Z,Pimpalgaon Baswant(Saykheda),Onion,1342.46
2026-03-29T00:00:00.000Z,Satana,Onion,1361.15
2026-03-29T00:00:00.000Z,Lasalgaon,Onion,1481.27
2026-03-29T00:00:00.000Z,Kalvan,Onion,1374.96
2026-03-29T00:00:00.000Z,Manmad,Onion,1303.15
